# ESID4HPO — semantic content & LIRICAL evaluation

**Goal.** Quantify what the ESID/HPO workshop added to the ontology by comparing patient
annotations expressed with two HPO releases:

| role | release | how it is obtained |
|------|---------|--------------------|
| **old** (before workshop) | `v2024-08-13` | phenopackets are *aged* down to this vocabulary |
| **new** (after workshop)  | `v2026-06-23` | phenopackets as curated |

The cohorts are curated from primary literature and annotated with the **new** HPO.
Two analyses are run over the same cohorts:

1. **Semantic content** — information content (IC) / specificity carried by the annotations,
   number of terms per patient, and the footprint of post-workshop terms.
2. **LIRICAL** — rank of the causal disease before vs. after, run via the `lirical benchmark`
   command with version-matched data bundles (`run_lirical_adjusted.py`).

Everything is driven by a `COHORTS` list and small reusable functions, so the same run is
replicated for **SOCS1**, **APDS1** and **NFKB1** by editing one config cell.

## 0  Setup & configuration

In [ ]:
import csv
import math
import os
import pathlib
import re
import typing
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import hpotk
from google.protobuf.json_format import Parse, MessageToJson
from phenopackets.schema.v2.phenopackets_pb2 import Phenopacket
from phenopackets.schema.v2.core.base_pb2 import OntologyClass
from phenopackets.schema.v2.core.phenotypic_feature_pb2 import PhenotypicFeature

try:
    from scipy.stats import wilcoxon
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    print("scipy not available - paired significance tests will be skipped")

### Paths

The notebook lives in `src/analysis/`. Cohorts are under `src/cohorts/<NAME>/phenopackets/`.
Outputs (aged phenopackets, LIRICAL inputs, figures) go under `src/analysis/_work/`.

In [ ]:
FPATH_NB   = pathlib.Path(os.getcwd())                 # .../src/analysis
FPATH_SRC  = FPATH_NB.parent                           # .../src
FPATH_COH  = FPATH_SRC / "cohorts"                     # .../src/cohorts
FPATH_WORK = FPATH_NB / "_work"                         # scratch + outputs
FPATH_FIGS = FPATH_WORK / "figures"
FPATH_LIRICAL = FPATH_WORK / "lirical"                 # LIRICAL inputs/outputs
for p in (FPATH_WORK, FPATH_FIGS, FPATH_LIRICAL):
    p.mkdir(parents=True, exist_ok=True)
FPATH_COH

### Cohorts

Add a cohort by dropping phenopackets in `src/cohorts/<NAME>/phenopackets/` and adding a row
here. `disease_id` is only used for labelling — LIRICAL determines the causal disease itself
(`is_causal` column of the benchmark output).

In [ ]:
@dataclass(frozen=True)
class Cohort:
    name: str
    disease_id: str
    disease_label: str
    color: str
    pp_subdir: str = "phenopackets"

    def pp_dir(self, root: pathlib.Path = FPATH_COH) -> pathlib.Path:
        return root / self.name / self.pp_subdir


COHORTS = [
    Cohort("SOCS1", "OMIM:619375",
           "Autoinflammatory syndrome, familial, with or without immunodeficiency",
           color="#2b6cb0"),
    Cohort("APDS1", "OMIM:615513",
           "Activated PI3K-delta syndrome", color="#dd6b20"),
    Cohort("NFKB1", "OMIM:616576",
           "Common variable immunodeficiency 12 (NFKB1)", color="#38a169"),
]

# keep only cohorts that actually have phenopackets on disk
ACTIVE = [c for c in COHORTS if c.pp_dir().is_dir() and any(c.pp_dir().glob("*.json"))]
print("Active cohorts:", [c.name for c in ACTIVE])
for c in COHORTS:
    n = len(list(c.pp_dir().glob("*.json"))) if c.pp_dir().is_dir() else 0
    print(f"  {c.name:6s} {n:3d} phenopackets")

### HPO releases

On a machine with internet access both releases are pulled from the HPO ontology store.
If you already have the JSON locally (e.g. an unreleased build), pass a path instead.

In [ ]:
HPO_OLD_TAG = "v2024-08-13"   # before the ESID4HPO workshop
HPO_NEW_TAG = "v2026-06-23"   # workshop release


def load_hpo(tag: str | None = None, path: str | pathlib.Path | None = None):
    """Load a MinimalOntology from the ontology store (by tag) or from a local json."""
    if path is not None:
        print(f"Loading HPO from {path}")
        return hpotk.load_minimal_ontology(str(path))
    store = hpotk.configure_ontology_store()
    return store.load_minimal_hpo(release=tag)


hpo_old = load_hpo(tag=HPO_OLD_TAG)
hpo_new = load_hpo(tag=HPO_NEW_TAG)
print("old:", hpo_old.version, " | new:", hpo_new.version)

## 1  Core functions

Everything below is plain functions so you can tweak thresholds, the IC definition or the
figure styling without touching the analysis flow.

### 1.1  Reading phenopackets

In [ ]:
def load_phenopackets(ppdir: pathlib.Path) -> list[tuple[str, Phenopacket]]:
    """Return [(filename, Phenopacket), ...] sorted by filename."""
    out = []
    for fp in sorted(ppdir.glob("*.json")):
        with fp.open() as fh:
            out.append((fp.name, Parse(fh.read(), Phenopacket())))
    return out


def feature_pairs(pp: Phenopacket, include_excluded: bool = True) -> list[tuple[str, bool]]:
    """(CURIE, excluded) for each phenotypic feature, de-duplicated, order preserved."""
    out, seen = [], set()
    for pf in pp.phenotypic_features:
        if pf.excluded and not include_excluded:
            continue
        key = (pf.type.id, pf.excluded)
        if key not in seen:
            seen.add(key); out.append(key)
    return out


def feature_curies(pp: Phenopacket, include_excluded: bool = True) -> list[str]:
    """Convenience: just the CURIEs (any status)."""
    return [c for c, _ in feature_pairs(pp, include_excluded)]

### 1.2  Aging annotations to the old vocabulary

Each term absent from the old release is replaced by its **nearest ancestor that exists in the
old release** (depth-first over `is_a` parents in the new graph, with a `seen` guard for the
multi-parent DAG). Terms already present in the old HPO are kept. Several new terms can
collapse onto one old ancestor — that loss of resolution is exactly what we want to measure.

**Excluded findings are handled separately.** A negated specific term cannot be moved up the
hierarchy (excluding *Decreased CD8+ TEMRA proportion* does not justify excluding *Abnormal
total T cell number*). Excluded features are kept only if directly representable in the old
release; otherwise they are dropped. Toggle `generalize_excluded=True` to override.

In [ ]:
def generalize_term(curie: str, hpo_old, hpo_new) -> str | None:
    """Nearest ancestor (incl. self) of `curie` present in `hpo_old`; None if untraceable."""
    if curie in hpo_old:
        return curie
    if curie not in hpo_new:
        return None
    seen: set[str] = set()
    stack = list(hpo_new.graph.get_parents(curie))
    while stack:
        cur = stack.pop()
        if cur.value in seen:
            continue
        seen.add(cur.value)
        if cur in hpo_old:
            return cur.value
        stack.extend(hpo_new.graph.get_parents(cur))
    return None


def age_feature(curie: str, excluded: bool, hpo_old, hpo_new,
                generalize_excluded: bool = False) -> str | None:
    """Age a single feature respecting its negation status.

    Observed findings generalize up to the nearest old-HPO ancestor. *Excluded* findings are
    NOT generalized by default: excluding a specific term does not license excluding its
    parent (that would assert the absence of every sibling too). An excluded finding therefore
    survives only if it is directly representable in the old release, otherwise it is dropped -
    which correctly captures negative information that the old vocabulary could not express.
    """
    if excluded and not generalize_excluded:
        return curie if curie in hpo_old else None
    return generalize_term(curie, hpo_old, hpo_new)

### 1.3  Information content (specificity)

Annotation-based **Resnik IC over the disease corpus**:
`IC(t) = -ln p(t)`, `p(t) = freq(t)/N`, where `freq(t)` counts *diseases* annotated to
`t` or to any descendant of `t`, and `N` is the number of diseases in `phenotype.hpoa`.
Normalised by `ln N` so that IC runs from 0 to 1. This is the diagnostic framing: how much
a term narrows the differential across all known diseases.

It requires the **augmented** HPOA (`_work/hpoa_adjusted/new/phenotype_augmented.hpoa`,
written by `run_lirical_adjusted.py`), because in the stock release the annotation lines for
the three cohort diseases are identical between v2024-08-13 and v2026-06-23: the workshop
terms have zero annotated diseases and their IC would be undefined.

`IC_METHOD` (section 2) selects one of two variants, both exposing `ic(curie) -> float | None`:

* **`"hpoa"`** — one frequency table from the full augmented HPOA.
* **`"hpoa_loo"`** (used for the manuscript) — leave-one-publication-out. A workshop term
  reaches near-maximal IC partly because *we* just annotated it, to 1–2 diseases out of
  ~13,000. Scoring each patient against an HPOA that excludes its own source PMID breaks that
  circularity. Both arms of a given patient still share one table, so the frozen-table rule
  below is kept.

> **Frozen table.** Both the **new** and the **aged** term sets are scored against one frequency
> table and one ontology graph (v2026-06-23), so the comparison isolates *which term was
> chosen*, not how the graph was re-shaped. Terms predating the workshop score identically in
> both arms; re-parenting and redefinition contribute no measured gain (conservative).

> **Direction.** IC is monotone non-increasing toward the root, so ageing can never *increase*
> it. The Wilcoxon test asks whether the collapse is non-trivial in magnitude, not whether it
> points the expected way.

In [ ]:
def make_ancestor_fn(hpo, root: str = "HP:0000118"):
    """Cached ancestor closure restricted to the `root` subtree."""
    cache: dict[str, set[str]] = {}

    def ancestors(curie: str) -> set[str]:
        hit = cache.get(curie)
        if hit is not None:
            return hit
        if curie not in hpo:
            cache[curie] = set()
            return cache[curie]
        anc = {a.value for a in hpo.graph.get_ancestors(curie)}
        anc.add(curie)
        anc = anc if (root in anc or curie == root) else set()
        cache[curie] = anc
        return anc

    return ancestors


def resnik_table(entity_term_sets, ancestors, normalize: bool = True):
    """IC(t) = -ln(freq(t)/n), freq counted over entities with ancestor closure."""
    n = len(entity_term_sets)
    if n == 0:
        raise ValueError("empty corpus")
    freq: dict[str, int] = {}
    for terms in entity_term_sets:
        expanded: set[str] = set()
        for c in terms:
            expanded |= ancestors(c)
        for t in expanded:
            freq[t] = freq.get(t, 0) + 1
    mx = math.log(n) if n > 1 else 1.0
    table = {t: (-math.log(min(f / n, 1.0))) / (mx if normalize else 1.0)
             for t, f in freq.items()}
    return table, freq, n


def load_hpoa_disease_terms(path, aspect: str = "P", drop_negated: bool = True,
                            only: set[str] | None = None) -> dict[str, set[str]]:
    """Parse a phenotype.hpoa into {disease_id: {hpo_curie, ...}}."""
    out: dict[str, set[str]] = {}
    with open(path, encoding="utf-8") as fh:
        rows = (ln for ln in fh if not ln.startswith("#"))
        for r in csv.DictReader(rows, delimiter="\t"):
            if aspect and r.get("aspect") != aspect:
                continue
            if drop_negated and (r.get("qualifier") or "").strip().upper() == "NOT":
                continue
            db, hp = r.get("database_id"), r.get("hpo_id")
            if not db or not hp:
                continue
            if only is not None and db not in only:
                continue
            out.setdefault(db, set()).add(hp)
    return out


def build_hpoa_ic(disease_terms, hpo, root: str = "HP:0000118",
                  normalize: bool = True):
    """Resnik IC over the disease corpus: freq(t) = number of DISEASES annotated
    to t or to any descendant of t (the diagnostic framing). Requires the
    augmented HPOA, otherwise the workshop terms have zero annotated diseases
    and their IC is undefined.
    """
    ancestors = make_ancestor_fn(hpo, root)
    table, freq, n = resnik_table(list(disease_terms.values()), ancestors, normalize)

    def ic(curie: str) -> float | None:
        return table.get(curie)

    ic.N = n              # type: ignore[attr-defined]
    ic.freq = freq        # type: ignore[attr-defined]
    ic.table = table      # type: ignore[attr-defined]
    ic.ontology = hpo     # type: ignore[attr-defined]
    return ic


class HpoaLooIC:
    """Leave-one-publication-out HPOA IC.

    A workshop term reaches near-maximal IC partly because we just annotated it
    to one disease. Scoring each patient against an HPOA that excludes its own
    source publication breaks that circularity. Both arms of a given patient
    still share one table, so the frozen-table rule is kept.

    Only the cohort diseases differ between the per-PMID files, so the base
    corpus is parsed once and only those diseases are swapped per PMID.
    """

    def __init__(self, base_path, loo_dir, cohort_disease_ids, hpo,
                 root="HP:0000118", normalize=True):
        self.hpo, self.root, self.normalize = hpo, root, normalize
        self.ancestors = make_ancestor_fn(hpo, root)
        self.base = load_hpoa_disease_terms(base_path)
        self.cohort_ids = set(cohort_disease_ids)
        self.loo_dir = pathlib.Path(loo_dir)
        self._tables: dict[str, dict] = {}
        self._active = self._table_for(None)

    def _table_for(self, pmid):
        key = pmid or "__full__"
        if key in self._tables:
            return self._tables[key]
        terms = dict(self.base)
        if pmid:
            f = self.loo_dir / f"phenotype_PMID_{pmid}.hpoa"
            if f.exists():
                swap = load_hpoa_disease_terms(f, only=self.cohort_ids)
                for d in self.cohort_ids:
                    if d in swap:
                        terms[d] = swap[d]
                    else:
                        terms.pop(d, None)
            else:
                print(f"    [warn] no LOO file for PMID:{pmid}, using full corpus")
        tbl, _, _ = resnik_table(list(terms.values()), self.ancestors, self.normalize)
        self._tables[key] = tbl
        return tbl

    def set_context(self, fname: str):
        m = re.search("PMID[_:]?([0-9]+)", str(fname))
        self._active = self._table_for(m.group(1) if m else None)

    def __call__(self, curie: str):
        return self._active.get(curie)

### 1.4  Per-cohort semantic table

In [ ]:
def semantic_tables(cohort: Cohort, hpo_old, hpo_new, ic,
                    include_excluded: bool = True):
    """Return (df_patient, df_terms) for one cohort.

    df_patient : one row / phenopacket with term counts and IC summaries (new vs aged).
    df_terms   : long form, one row / (phenopacket, term, version) with its IC.
    """
    prows, trows = [], []
    for fname, pp in load_phenopackets(cohort.pp_dir()):
        if hasattr(ic, "set_context"):
            ic.set_context(fname)   # leave-one-PMID-out: swap the active table
        pairs = feature_pairs(pp, include_excluded)
        new_curies = list(dict.fromkeys(c for c, _ in pairs))
        aged_seen, aged_curies = set(), []
        for c, exc in pairs:
            g = age_feature(c, exc, hpo_old, hpo_new)
            if g is not None and g not in aged_seen:
                aged_seen.add(g); aged_curies.append(g)

        new_ic  = [ic(c) for c in new_curies  if ic(c) is not None]
        aged_ic = [ic(c) for c in aged_curies if ic(c) is not None]
        n_newly = sum(1 for c in new_curies if c not in hpo_old)

        prows.append(dict(
            cohort=cohort.name, phenopacket=fname,
            n_new=len(new_curies), n_aged=len(aged_curies),
            # terms with no annotated disease in the (leave-one-out) corpus have
            # undefined IC and are excluded from the means; counted here (Table S2)
            n_ic_undefined_new=len(new_curies) - len(new_ic),
            n_ic_undefined_aged=len(aged_curies) - len(aged_ic),
            ic_new=float(np.sum(new_ic)),  ic_aged=float(np.sum(aged_ic)),
            mean_ic_new=float(np.mean(new_ic))  if new_ic  else np.nan,
            mean_ic_aged=float(np.mean(aged_ic)) if aged_ic else np.nan,
            n_newly_enabled=n_newly,
            frac_newly_enabled=n_newly / len(new_curies) if new_curies else np.nan,
        ))
        for c in new_curies:
            if ic(c) is not None:
                trows.append(dict(cohort=cohort.name, phenopacket=fname,
                                  version="new", curie=c, ic=ic(c)))
        for c in aged_curies:
            if ic(c) is not None:
                trows.append(dict(cohort=cohort.name, phenopacket=fname,
                                  version="old", curie=c, ic=ic(c)))
    return pd.DataFrame(prows), pd.DataFrame(trows)

### 1.5  LIRICAL benchmark I/O

In [ ]:
def load_lirical_ranks(csv_path: pathlib.Path) -> pd.DataFrame:
    """Causal-disease rank per phenopacket from a `lirical benchmark` CSV."""
    df = pd.read_csv(csv_path)
    causal = df.loc[df["is_causal"], ["phenopacket", "rank", "post_test_proba"]]
    return causal.set_index("phenopacket").sort_index()


def lirical_rank_table(old_csv, new_csv,
                       old_label="old", new_label="new") -> pd.DataFrame:
    """Wide table: index=phenopacket, columns=[old_label, new_label] of causal rank."""
    old = load_lirical_ranks(pathlib.Path(old_csv))["rank"].rename(old_label)
    new = load_lirical_ranks(pathlib.Path(new_csv))["rank"].rename(new_label)
    return pd.concat([old, new], axis=1)

### 1.6  Figure styling

In [ ]:
def set_style():
    mpl.rcParams.update({
        "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
        "font.size": 11, "font.family": "DejaVu Sans",
        "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 11,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "legend.frameon": False, "legend.fontsize": 10,
    })

COL_OLD = "#9aa0a6"
COL_NEW = "#2b6cb0"
COHORT_COLOR = {c.name: c.color for c in COHORTS}

set_style()

### 1.7  LIRICAL figures

In [ ]:
def plot_lirical_slope(rank_table, cohort_name, old_label="old", new_label="new", ax=None):
    """Slopegraph of causal-disease rank per patient, old -> new (improved lines highlighted)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.0, 4.4))
    for o, n in zip(rank_table[old_label], rank_table[new_label]):
        if pd.isna(o) or pd.isna(n):
            continue
        ax.plot([0, 1], [o, n], color=(COL_NEW if n < o else COL_OLD),
                alpha=.55, lw=1.0, zorder=1)
    ax.scatter(np.zeros(len(rank_table)), rank_table[old_label],
               s=22, color=COL_OLD, edgecolor="w", lw=.4, zorder=3)
    ax.scatter(np.ones(len(rank_table)), rank_table[new_label],
               s=22, color=COL_NEW, edgecolor="w", lw=.4, zorder=3)
    ax.invert_yaxis()
    ax.set_xticks([0, 1]); ax.set_xticklabels([f"Old\n{HPO_OLD_TAG}", f"New\n{HPO_NEW_TAG}"])
    ax.set_xlim(-.3, 1.3)
    ax.set(ylabel="Rank of causal disease (lower = better)", title=cohort_name)
    return ax


def plot_lirical_summary_box(df_ranks_long, ax=None):
    """Paired boxes (old vs new) of causal rank for every cohort."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.4, 4.2))
    cohorts = list(dict.fromkeys(df_ranks_long.cohort))
    data, pos, cols, tickpos = [], [], [], []
    for i, ch in enumerate(cohorts):
        s = df_ranks_long[df_ranks_long.cohort == ch]
        data += [s[s.version == "old"]["rank"].values, s[s.version == "new"]["rank"].values]
        pos += [i*2.6 + 1, i*2.6 + 1.8]; cols += [COL_OLD, COL_NEW]
        tickpos.append(i*2.6 + 1.4)
    bp = ax.boxplot(data, positions=pos, widths=.7, patch_artist=True, showfliers=False)
    for b, c in zip(bp["boxes"], cols):
        b.set(facecolor=c, alpha=.6, edgecolor="#333", lw=1.0)
    for w in bp["whiskers"] + bp["caps"]:
        w.set(color="#333", lw=1.0)
    for m in bp["medians"]:
        m.set(color="#111", lw=1.6)
    ax.set_xticks(tickpos); ax.set_xticklabels(cohorts)
    ax.invert_yaxis()
    ax.set(ylabel="Rank of causal disease (lower = better)",
           title="LIRICAL ranking: old vs new HPO")
    ax.legend(handles=[mpl.patches.Patch(facecolor=COL_OLD, alpha=.6, label=f"Old ({HPO_OLD_TAG})"),
                       mpl.patches.Patch(facecolor=COL_NEW, alpha=.6, label=f"New ({HPO_NEW_TAG})")])
    return ax


def plot_topk_recall(df_ranks_long, ks=(1, 3, 5, 10), ax=None):
    """Top-k recall of the causal disease, old (dashed) vs new (solid), per cohort."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.4, 4.2))
    cohorts = list(dict.fromkeys(df_ranks_long.cohort))
    xs = np.arange(len(ks))
    for ch in cohorts:
        s = df_ranks_long[df_ranks_long.cohort == ch]
        old = [(s[s.version == "old"]["rank"] <= k).mean() * 100 for k in ks]
        new = [(s[s.version == "new"]["rank"] <= k).mean() * 100 for k in ks]
        col = COHORT_COLOR.get(ch, COL_NEW)
        ax.plot(xs, old, "o--", color=col, alpha=.5, mfc="white", label=f"{ch} old")
        ax.plot(xs, new, "o-",  color=col, label=f"{ch} new")
    ax.set_xticks(xs); ax.set_xticklabels([f"Top-{k}" for k in ks])
    ax.set(ylabel="% cases with causal disease recalled",
           title="Top-k recall: old (dashed) vs new (solid)")
    ax.set_ylim(0, 105); ax.legend(ncol=max(1, len(cohorts)), fontsize=8)
    return ax

## 2  Semantic-content analysis

Build one IC scorer on the **new** ontology and compute the semantic tables for every active
cohort.

In [ ]:
# --- IC method -------------------------------------------------------------
# "hpoa"     : Resnik IC over the augmented disease corpus, one frequency table.
# "hpoa_loo" : same, but each patient is scored against an HPOA excluding its
#              own source publication (manuscript setting; slower).
# Both need _work/hpoa_adjusted/new/ from run_lirical_adjusted.py.
IC_METHOD    = "hpoa_loo"
IC_NORMALIZE = True

HPOA_AUG_DIR   = FPATH_WORK / "hpoa_adjusted" / "new"
HPOA_AUG       = HPOA_AUG_DIR / "phenotype_augmented.hpoa"
COHORT_DISEASE = {c.name: c.disease_id for c in COHORTS}

assert HPOA_AUG.exists(), f"missing {HPOA_AUG} - run run_lirical_adjusted.py first"
if IC_METHOD == "hpoa":
    dt = load_hpoa_disease_terms(HPOA_AUG)
    ic_hpoa = build_hpoa_ic(dt, hpo_new, normalize=IC_NORMALIZE)
    print(f"  HPOA corpus: {ic_hpoa.N} diseases, {len(ic_hpoa.table)} scored terms")
    for name, did in COHORT_DISEASE.items():
        print(f"    {name:6s} {did}  {len(dt.get(did, ())):4d} direct P terms")
    ic_by_cohort = {c.name: ic_hpoa for c in ACTIVE}
elif IC_METHOD == "hpoa_loo":
    loo = HpoaLooIC(HPOA_AUG, HPOA_AUG_DIR, COHORT_DISEASE.values(),
                    hpo_new, normalize=IC_NORMALIZE)
    print(f"  HPOA corpus: {len(loo.base)} diseases, leave-one-publication-out active")
    ic_by_cohort = {c.name: loo for c in ACTIVE}
else:
    raise ValueError(IC_METHOD)

### 2.1  Sanity check — are all annotated terms known?

Every term must be in the new HPO. Terms missing from the old HPO are the ones the workshop
added (or that were otherwise introduced after `v2024-08-13`).

In [ ]:
unknown_new, missing_old = set(), set()
for c in ACTIVE:
    for _, pp in load_phenopackets(c.pp_dir()):
        for curie in feature_curies(pp):
            if curie not in hpo_new:
                unknown_new.add(curie)
            if curie not in hpo_old:
                missing_old.add(curie)

print(f"Terms unknown to NEW HPO (should be empty): {len(unknown_new)}")
for u in sorted(unknown_new):
    print("   ", u)
print(f"\nTerms absent from OLD HPO (post-{HPO_OLD_TAG}): {len(missing_old)}")
for u in sorted(missing_old)[:25]:
    print(f"    {u}  {hpo_new.get_term_name(u)}")

### 2.2  Compute tables

In [ ]:
patient_frames, term_frames = [], []
for c in ACTIVE:
    dfp, dft = semantic_tables(c, hpo_old, hpo_new, ic_by_cohort[c.name], include_excluded=True)
    patient_frames.append(dfp); term_frames.append(dft)

df_patient = pd.concat(patient_frames, ignore_index=True)
df_terms   = pd.concat(term_frames,   ignore_index=True)

summary = (df_patient.groupby("cohort")
           .agg(n_patients=("phenopacket", "nunique"),
                mean_terms_new=("n_new", "mean"),
                mean_terms_aged=("n_aged", "mean"),
                mean_ic_new=("mean_ic_new", "mean"),
                mean_ic_aged=("mean_ic_aged", "mean"),
                total_ic_new=("ic_new", "mean"),
                total_ic_aged=("ic_aged", "mean"),
                pct_post_workshop=("frac_newly_enabled", lambda s: s.mean()*100))
           .round(3))
summary

In [ ]:
# Figure 3 - information content. Panel A: paired per-patient IC per cohort;
# panel C: fraction of annotations using post-workshop terms. The plotting code
# lives in figures/fig3_panels.py and runs in this namespace (it needs
# df_patient, FPATH_FIGS, FPATH_WORK, HPO_*_TAG). It also writes
# _work/fig3_ic_stats.csv and leaves `stats_tbl` behind for the table below.
FIG3_PATH = FPATH_NB / "figures" / "fig3_panels.py"
exec(open(FIG3_PATH).read())

In [ ]:
# Table 1 (manuscript): per-cohort summary of the semantic analysis.
t = summary.join(stats_tbl.set_index("cohort")).loc[[c.name for c in ACTIVE]]

# raw p per cohort: exact signed-rank p when few informative pairs all share one
# sign (two-sided p = 2 / 2**n_inf); otherwise the scipy value.
p_raw, exact = [], []
for ni, im, wo, pr in zip(t.n_informative, t.n_improved, t.n_worse, t.p_raw):
    if ni < 10 and (im == 0 or wo == 0):
        p_raw.append(2.0 / 2 ** int(ni))
        exact.append(True)
    else:
        p_raw.append(float(pr))
        exact.append(False)

# Holm step-down across cohorts, on the corrected raw p's
m = len(p_raw)
order = np.argsort(p_raw)
holm_sorted = np.minimum(1.0, np.maximum.accumulate(
    [(m - k) * p_raw[i] for k, i in enumerate(order)]))
p_holm = np.empty(m)
p_holm[order] = holm_sorted

SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")

def fmt_p(p, is_exact):
    if is_exact:
        s = f"{p:.4f}".rstrip("0")          # 0.0625 stays exact, no fake precision
    elif p >= 0.001:
        s = f"{p:.3f}"
    else:
        mant, exp = f"{p:.1e}".split("e")
        s = f"{mant} × 10{str(int(exp)).translate(SUP)}"
    return s + (" (exact)" if is_exact else "")

tbl = pd.DataFrame({
    "n": t.n.astype(int),
    "terms/indiv. (pre→post)": [f"{x:.1f} → {y:.1f}"
                                for x, y in zip(t.mean_terms_aged, t.mean_terms_new)],
    "mean IC (pre→post)":      [f"{x:.3f} → {y:.3f}"
                                for x, y in zip(t.mean_ic_aged, t.mean_ic_new)],
    "% post-workshop terms": t.pct_post_workshop.round(1),
    "ΔIC [95% CI]": [f"+{d:.3f} [{lo:.3f}, {hi:.3f}]"
                     for d, lo, hi in zip(t.delta_mean, t.delta_lo, t.delta_hi)],
    "improved/worse": [f"{int(i)}/{int(w)}"
                       for i, w in zip(t.n_improved, t.n_worse)],
    "p (Holm)": [fmt_p(p, e) for p, e in zip(p_holm, exact)],
}, index=t.index)
tbl.to_csv(FPATH_WORK / "table1_manuscript.csv")

# Supplementary Table S2: terms per individual whose IC is undefined in the
# leave-one-publication-out corpus (no annotated disease left) and were excluded
tblS2 = (df_patient.groupby("cohort")
         .agg(n=("phenopacket", "nunique"),
              terms_post=("n_new", "sum"), undefined_post=("n_ic_undefined_new", "sum"),
              terms_pre=("n_aged", "sum"), undefined_pre=("n_ic_undefined_aged", "sum"),
              individuals_affected=("n_ic_undefined_new", lambda s: int((s > 0).sum())))
         .loc[[c.name for c in ACTIVE]])
tblS2.to_csv(FPATH_WORK / "tableS2_ic_undefined.csv")
display(tblS2)
tbl

## 3  Prepare phenopackets for LIRICAL

For each cohort two phenopacket sets are staged under `_work/lirical/<COHORT>/`:

* **new/** — the curated phenopackets, unchanged.
* **old/** — the *aged* phenopackets.

The aged phenopacket keeps the original disease/genomic interpretation but replaces the
phenotypic features with their aged CURIEs. `run_lirical_adjusted.py` (or `rerun_all.sh`)
then augments the HPOA with the cohort annotations, builds the leave-one-publication-out
bundles, and runs `lirical benchmark` on both sets.

In [ ]:
def write_aged_phenopacket(pp, hpo_old, hpo_new) -> Phenopacket:
    """Copy a phenopacket, replacing phenotypic features with aged terms (keeps excluded flag)."""
    aged = Phenopacket(); aged.CopyFrom(pp)
    aged.phenotypic_features.clear()
    mapped: dict[str, bool] = {}      # curie -> excluded (observed wins on collision)
    for pf in pp.phenotypic_features:
        g = age_feature(pf.type.id, pf.excluded, hpo_old, hpo_new)
        if g is None:
            continue
        if g in mapped:
            mapped[g] = mapped[g] and pf.excluded   # observed (False) wins
        else:
            mapped[g] = pf.excluded
    for g, exc in mapped.items():
        aged.phenotypic_features.append(PhenotypicFeature(
            type=OntologyClass(id=g, label=hpo_new.get_term_name(g)), excluded=exc))
    return aged


def stage_lirical_inputs(cohort: Cohort, hpo_old, hpo_new, root=FPATH_LIRICAL):
    new_dir = root / cohort.name / "new"; new_dir.mkdir(parents=True, exist_ok=True)
    old_dir = root / cohort.name / "old"; old_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for fname, pp in load_phenopackets(cohort.pp_dir()):
        (new_dir / fname).write_text(MessageToJson(pp))
        (old_dir / fname).write_text(MessageToJson(write_aged_phenopacket(pp, hpo_old, hpo_new)))
        n += 1
    print(f"{cohort.name:6s}  staged {n} phenopackets -> {new_dir.parent}")
    return old_dir, new_dir


for c in ACTIVE:
    stage_lirical_inputs(c, hpo_old, hpo_new)

## 4  LIRICAL evaluation

Run `bash rerun_all.sh` (or `python run_lirical_adjusted.py`) from `src/analysis/`, then load
the merged benchmark CSVs below. The next cell tolerates missing files so the notebook can be
developed before LIRICAL has finished.

In [ ]:
rank_tables, rank_long_rows = {}, []
for c in ACTIVE:
    old_csv = FPATH_LIRICAL / f"{c.name}.old.csv"
    new_csv = FPATH_LIRICAL / f"{c.name}.new.csv"
    if not (old_csv.exists() and new_csv.exists()):
        print(f"{c.name:6s}  (LIRICAL output not found yet - run run_lirical_adjusted.py)")
        continue
    rt = lirical_rank_table(old_csv, new_csv)
    rank_tables[c.name] = rt
    for fname, row in rt.dropna().iterrows():
        rank_long_rows.append(dict(cohort=c.name, phenopacket=fname, version="old", rank=row["old"]))
        rank_long_rows.append(dict(cohort=c.name, phenopacket=fname, version="new", rank=row["new"]))

df_ranks = pd.DataFrame(rank_long_rows)
print("loaded cohorts:", list(rank_tables))

### 4.1  Rank table & improvement stats (Table 2)

Paired Wilcoxon signed-rank per cohort, matched-pairs rank-biserial effect size and Holm
correction across cohorts (`_rank_biserial` / `_holm` come from `fig3_panels.py`, run in
section 2). Exported unrounded to `_work/table2_lirical.csv`.

In [ ]:
if rank_tables:
    rows = []
    for ch, rt in rank_tables.items():
        d = rt.dropna()
        improved = (d["new"] < d["old"]).sum()
        worse    = (d["new"] > d["old"]).sum()
        line = dict(cohort=ch, n=len(d),
                    median_old=d["old"].median(), median_new=d["new"].median(),
                    top1_old=(d["old"] <= 1).mean()*100, top1_new=(d["new"] <= 1).mean()*100,
                    top10_old=(d["old"] <= 10).mean()*100, top10_new=(d["new"] <= 10).mean()*100,
                    improved=improved, worse=worse)
        if HAVE_SCIPY and len(d) >= 5 and (d["new"] - d["old"]).abs().sum() > 0:
            line["wilcoxon_p"] = wilcoxon(d["new"], d["old"]).pvalue
            # matched-pairs rank-biserial; sign flipped so that a lower (better)
            # rank in the workshop arm gives a positive effect size
            line["rank_biserial"] = _rank_biserial(d["old"], d["new"])
        rows.append(line)
    lirical_stats = pd.DataFrame(rows)
    if "wilcoxon_p" in lirical_stats:
        m = lirical_stats.wilcoxon_p.notna()
        lirical_stats["p_holm"] = np.nan
        lirical_stats.loc[m, "p_holm"] = _holm(lirical_stats.loc[m, "wilcoxon_p"].values)
    lirical_stats.to_csv(FPATH_WORK / "table2_lirical.csv", index=False)
    # round for display only; the CSV keeps full precision
    display(lirical_stats.round({c: 2 for c in lirical_stats.columns
                                 if c not in ("wilcoxon_p", "p_holm", "rank_biserial")}))
else:
    print("No LIRICAL results loaded yet.")

### 4.2  Figures — LIRICAL

In [ ]:
if rank_tables:
    n = len(rank_tables)
    fig, axes = plt.subplots(1, n, figsize=(4.0 * n, 4.6), squeeze=False)
    for ax, (ch, rt) in zip(axes[0], rank_tables.items()):
        plot_lirical_slope(rt, ch, ax=ax)
    fig.suptitle("Causal-disease rank per individual: old -> new",
                 fontsize=13, fontweight="bold", y=1.03)
    fig.tight_layout()
    fig.savefig(FPATH_FIGS / "lirical_per_cohort.png")

In [ ]:
if not df_ranks.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    plot_lirical_summary_box(df_ranks, ax=axes[0])
    plot_topk_recall(df_ranks, ax=axes[1])
    fig.suptitle("LIRICAL improvement across cohorts",
                 fontsize=13, fontweight="bold", y=1.04)
    fig.tight_layout()
    fig.savefig(FPATH_FIGS / "lirical_summary.png")
    fig.savefig(FPATH_FIGS / "lirical_summary.pdf")

## 5  Export tables

In [ ]:
df_patient.to_csv(FPATH_WORK / "semantic_patient_level.csv", index=False)
df_terms.to_csv(FPATH_WORK / "semantic_term_level.csv", index=False)
summary.to_csv(FPATH_WORK / "semantic_summary.csv")
if not df_ranks.empty:
    df_ranks.to_csv(FPATH_WORK / "lirical_ranks_long.csv", index=False)
print("Wrote tables and figures to", FPATH_WORK)